In [2]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 47.4 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=c45b7ba688a7a53f431d3203a7f10400a96d3ae34fd563f48b6e57b46289ba7d
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=a5bfb794e66edf11925f07b5b3a44560106990e3ced2d0ff7f465558c7711374
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel f

In [4]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
# We have to remind the new program about this little trick!
import sys
import torchvision.transforms.functional as F_vision
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive

from torchvision.models.video import swin3d_t
from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
# Connect to Google Drive so we can grab the "brain" and the video
drive.mount('/content/drive', force_remount=True)

CLASSES = ['walking', 'running']
NUM_FRAMES = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Here are the paths we need!
MODEL_WEIGHTS_PATH = "/content/drive/MyDrive/video_swin3d_TrainedModel.pth"
VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"

# --- 3. BUILD THE ROBOT BODY & LOAD THE BRAIN ---
print("Building the robot body...")
# 1. Create the empty Swin3D model
model = swin3d_t()

# 2. Change the head so it knows we only have 2 classes (walking, running)
model.head = nn.Linear(model.head.in_features, len(CLASSES))

# 3. Put the trained brain into the empty body!
print("Loading the trained brain from Google Drive...")
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))

# 4. Move it to the GPU and tell it we are taking a test, NOT training
model = model.to(DEVICE)
model.eval()

# --- 4. VIDEO GLASSES (TRANSFORMS) ---
# The model needs to view the video exactly the same way it did during training
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0), # Normalize to 0-1
            NormalizeVideo(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            Resize((224, 224))
        ])
    )
])

# --- 5. INFERENCE (THE MAGIC TRICK) ---
def see_and_guess(video_path):
    print("\nGetting the popcorn... loading video!")
    try:
        # Load the 2-second clip
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)

        # Put the glasses on the video (apply transforms)
        video_data = video_transform(video_data)

        # Add a batch dimension (the model expects a group, even if it's a group of 1)
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        # Tell the model to guess without trying to learn/train (saves memory!)
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")
        print(f"THE ROBOT GUESSES: {CLASSES[pred_idx.item()].upper()}")
        print(f"HOW SURE IS IT?: {conf.item()*100:.2f}%")
        print("-" * 30)

    except Exception as e:
        print(f"Oh no! Something went wrong: {e}")

# Let's run it!
see_and_guess(VIDEO_PATH)

Mounted at /content/drive
Building the robot body...
Loading the trained brain from Google Drive...

Getting the popcorn... loading video!
------------------------------
FILE: v_PlayingCello_g04_c02.avi
THE ROBOT GUESSES: RUNNING
HOW SURE IS IT?: 99.87%
------------------------------
